# `Материалы кафедры ММП факультета ВМК МГУ. Введение в Эффективные Системы Глубокого Обучения.`

# `Семинар 04. Линейное внимание`

### `Материалы составил Алексеев Илья (@voorhs)`

#### `Москва, Осенний семестр 2026`

О чём можно узнать из этого ноутбука:

* три формы линейного внимания — параллельная, рекуррентная и чанковая —
  вычисляют одну функцию, и референсный слой Qwen3-Next в `transformers`
  вычисляет её же;
* почему форма с *меньшим* числом FLOP медленнее на один-два порядка;
* как чанковая форма записывается четырьмя операторами Triton, которые вы
  напишете сами и проверите против PyTorch;
* почему декодирование выходит на плато, пока KV-кэш растёт.

**Обозначения** (одни и те же во всём модуле):

$$q,k\in\mathbb{R}^{d_k}\qquad v\in\mathbb{R}^{d_v}\qquad S\in\mathbb{R}^{d_k\times d_v}\qquad o_t=S_t^{\top}q_t$$

$$S_t=\sum_{j\le t}k_jv_j^{\top}\quad\quad\text{(post-update: }o_t\text{ читает состояние после того, как записан токен }t\text{)}$$

`q, k` имеют форму `(B, H, N, d_k)`; `v` и `o` — `(B, H, N, d_v)`; состояние —
`(B, H, d_k, d_v)`. `d_k` и `d_v` независимы — никогда не отождествляйте их.

### `Подготовка`

Запустите ячейку ниже первой и запускайте заново
после любого перезапуска ядра. Ни один шаг не читает переменных, определённых в другом
шаге, так что после ячейки 1 можно сразу перейти к тому шагу, на котором остановились
в прошлый раз.

In [ ]:
import importlib
import importlib.metadata as _md
import math, os, subprocess, sys, time, warnings
from pathlib import Path

import torch
import torch.nn.functional as F

# Библиотека курса. Три источника, в этом порядке: уже установленная, исходники
# рядом с ноутбуком, установка с GitHub. Порядок важен: тот, кто правит `src/`,
# должен запускать свою копию, а не скачанную.
LINATTN_SPEC = "linattn @ git+https://github.com/voorhs/linear-attention-public@v0.2.0"

def _import_linattn():
    try:
        return importlib.import_module("linattn")
    except ImportError:
        pass

    here = Path.cwd().resolve()                       # запуск из клона репозитория курса
    for candidate in (here, *here.parents):
        if (candidate / "src" / "linattn" / "__init__.py").exists():
            sys.path.insert(0, str(candidate / "src"))
            try:
                return importlib.import_module("linattn")
            except ImportError:
                sys.path.pop(0)
            break

    print(f"устанавливаем {LINATTN_SPEC} ...")        # Kaggle, Colab: своей копии нет
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                           LINATTN_SPEC], capture_output=True, text=True)
    if done.returncode:
        raise ImportError(
            "не удалось установить linattn. Если это Kaggle, включите "
            "Settings -> Internet (нужен аккаунт с подтверждённым телефоном).\n"
            + (done.stderr or done.stdout).strip()[-400:]) from None
    importlib.invalidate_caches()
    return importlib.import_module("linattn")

linattn = _import_linattn()

from linattn import (bench, chunkwise_linear_attention, decode,
                     parallel_linear_attention, plotting, prefill,
                     recurrent_linear_attention, step as decode_step)

%matplotlib inline

# единая конфигурация, используемая во всём ноутбуке
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float32     # bf16 нет в T4 (sm_75).
B, H, D_K, D_V = 1, 8, 64, 64
CHUNK = 128                # около измеренного минимума U-кривой — см. шаг 2
SEED  = 0
torch.manual_seed(SEED)

def inputs(n, rule="linear", *, b=B, h=H, d_k=D_K, d_v=D_V, chunk_size=CHUNK,
           seed=SEED, device=None):
    '''Детерминированные q, k, v (+ decay/beta) — генерируются на CPU, затем переносятся.

    Одни и те же числа на любом устройстве, так что кривая, измеренная на ноутбуке,
    и кривая, измеренная на GPU в аудитории, — кривые над одними и теми же входами.
    '''
    key = bench.RunKey(form="chunkwise", rule=rule, b=b, h=h, n=n, d_k=d_k,
                       d_v=d_v, chunk_size=chunk_size, dtype="float32",
                       device=str(device or DEVICE), seed=seed)
    return bench.make_bench_inputs(key)

TOLERANCE_FACTOR = 16.0

def scaled_tolerances(expected, n, factor=TOLERANCE_FACTOR):
    eps   = float(torch.finfo(expected.dtype).eps)
    rtol  = factor * eps * math.sqrt(max(n, 1))
    scale = float(expected.detach().abs().max()) if expected.numel() else 1.0
    return rtol, rtol * max(scale, 1.0)

def residual(actual, expected, n):
    '''(макс. абс. разность, atol, с которым она сравнивается, прошла ли проверка).'''
    rtol, atol = scaled_tolerances(expected, n)
    diff = float((actual - expected).abs().max())
    return diff, atol, bool(torch.allclose(actual, expected, rtol=rtol, atol=atol))

def assert_close(actual, expected, n, what):
    diff, atol, ok = residual(actual, expected, n)
    if not ok:
        raise AssertionError(f"{what}: max|Δ| = {diff:.3e} > atol {atol:.3e} при N={n}")
    head = "точно" if diff == 0 else f"запас {atol / diff:,.0f}×"
    print(f"  PASS  {what}:  max|Δ| = {diff:.2e}   (atol {atol:.2e}, {head})")

# softmax-бейзлайн на flash/memory-efficient ядрах, на тех же осях
from torch.nn.attention import SDPBackend, sdpa_kernel

def _probe_sdpa():
    probe = torch.randn(1, 2, 128, 32, device=DEVICE, dtype=DTYPE)
    good = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for backend in (SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION):
            try:
                with sdpa_kernel(backend):
                    F.scaled_dot_product_attention(probe, probe, probe, is_causal=True)
                good.append(backend)
            except Exception:
                pass
    return good

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

SDPA_BACKENDS = _probe_sdpa() if DEVICE.type == "cuda" else []
SOFTMAX_IS_SUBQUADRATIC = bool(SDPA_BACKENDS)

def _run_softmax(bench_inputs):
    q, k, v = bench_inputs.q, bench_inputs.k, bench_inputs.v
    if SDPA_BACKENDS:
        with sdpa_kernel(SDPA_BACKENDS):
            return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, is_causal=True)

def _softmax_bytes(key):
    elem = bench.dtype_bytes(key.dtype)
    io = key.b * key.h * key.n * (2 * key.d_k + 2 * key.d_v) * elem
    if SOFTMAX_IS_SUBQUADRATIC:
        return 2 * io                                   # O(N): q,k,v,out и немного сверху
    return io + 2 * key.b * key.h * key.n * key.n * elem  # math-бэкенд: O(N^2)

bench.register_form(bench.BenchForm(
    name="softmax", run=_run_softmax, predict_bytes=_softmax_bytes,
    rules=("linear",),   # это не правило обновления; перебору просто нужна одна метка
))

# платим за компиляцию Triton сейчас, а не посреди упражнения
if DEVICE.type == "cuda" and linattn.triton_is_available():
    _wq = torch.randn(1, 1, 64, D_K, device=DEVICE, dtype=DTYPE)
    _wv = torch.randn(1, 1, 64, D_V, device=DEVICE, dtype=DTYPE)
    _t0 = time.perf_counter()
    linattn.triton_chunkwise_linear_attention(_wq, _wq, _wv, chunk_size=32)
    torch.cuda.synchronize()
    print(f"прогрев triton (L=32, d_k=d_v={D_K}): {time.perf_counter() - _t0:.2f} s")
    del _wq, _wv

print(f"linattn  {Path(linattn.__file__).parent}")
print(f"устройство {DEVICE}   dtype {DTYPE}   B={B} H={H} d_k={D_K} d_v={D_V}   "
      f"чанк {CHUNK}   seed {SEED}")
print("softmax-бейзлайн субквадратичен по памяти:", SOFTMAX_IS_SUBQUADRATIC,
      [b.name for b in SDPA_BACKENDS])

### `Шаг 0. Машина, пины и единственный dtype`

In [ ]:
# пины версий
PINS = {"torch": "2.14.0", "triton": "3.8.0", "transformers": "5.16.1"}

# Поставьте False, если у этой машины нет сети.
ALLOW_INSTALL = True

def installed(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def base_version(v):
    return None if v is None else v.split("+")[0]

for pkg, want in PINS.items():
    have = installed(pkg)
    mark = "ok " if base_version(have) == want else "!! "
    print(f"{mark}{pkg:<14} нужно {want:<10} есть {have}")
    if base_version(have) == want:
        continue
    if pkg == "torch":
        # torch не переустанавливаем: это гигабайты и перезапуск ядра.
        print(f"   torch {have} != зафиксированной {want}. Посреди занятия не переустанавливаем. "
              f"Если числа ниже выглядят странно, это первый подозреваемый.")
        continue
    if not ALLOW_INSTALL:
        print(f"   ALLOW_INSTALL = False; пропускаем {pkg}=={want}")
        continue
    print(f"   устанавливаем {pkg}=={want} ...")
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                           f"{pkg}=={want}"], capture_output=True, text=True)
    if done.returncode:
        print(f"   НЕ УДАЛОСЬ УСТАНОВИТЬ {pkg}=={want}. Если это Kaggle, включите "
              f"Settings -> Internet (нужен аккаунт с подтверждённым телефоном).")
        print("   " + (done.stderr or done.stdout).strip().splitlines()[-1][:160])

In [ ]:
# на чём мы на самом деле работаем
if DEVICE.type == "cuda":
    prop = torch.cuda.get_device_properties(0)
    print(f"{prop.name}   sm_{prop.major}{prop.minor}   {prop.multi_processor_count} SM   "
          f"{prop.total_memory / 1024**3:.1f} GB")
    shared = getattr(prop, "shared_memory_per_multiprocessor",
                     prop.shared_memory_per_block)
    print(f"разделяемая память на SM: {shared / 1024:.0f} KiB.  Тайлинг ядра проверяется "
          f"против {linattn.TURING_SHARED_MEMORY_BYTES // 1024} KiB у T4 — там живёт шаг 4.")
print(bench.device_memory_budget(DEVICE).describe())
print("triton доступен:", linattn.triton_is_available())

In [ ]:
# окружение такое, каким его считает остальной ноутбук
assert DEVICE.type == "cuda", "семинару нужно CUDA-устройство (ядро написано на Triton)"
assert DTYPE is torch.float32, "только fp32: bf16 запрещён на sm_75"
assert linattn.triton_is_available(), "triton не импортируется"

bad = torch.zeros(1, 1, 32, D_K, device=DEVICE, dtype=torch.bfloat16)
bad_v = torch.zeros(1, 1, 32, D_V, device=DEVICE, dtype=torch.bfloat16)
try:
    linattn.triton_chunkwise_linear_attention(bad, bad, bad_v, chunk_size=32)
except linattn.KernelDomainError as error:
    print("  PASS  bfloat16 отвергнут:", str(error).split(".")[0])
else:
    raise AssertionError("ядро приняло bfloat16 — запрет для sm_75 не работает")
del bad, bad_v
print("  PASS  окружение")

### `Шаг 1. Два взгляда, одна функция — и разрыв по времени выполнения`

Softmax-внимание вычисляет $o_t=\sum_{j\le t}\frac{\exp(q_t^{\top}k_j)}{\sum_{j'}\exp(\cdot)}v_j$.
Уберите экспоненту и знаменатель — и можно перегруппировать:

$$o_t=\sum_{j\le t}(q_t^{\top}k_j)\,v_j=q_t^{\top}\underbrace{\sum_{j\le t}k_jv_j^{\top}}_{S_t}$$

Слева: матрица оценок $N\times N$ — **параллельная форма**, $O(N^2d)$. Справа: одна
матрица $d_k\times d_v$, которую несут вперёд, — **рекуррентная форма**, $O(Nd^2)$
и $N$ последовательных шагов. Одни и те же числа. Две программы.

#### `Задание 1. Параллельная форма линейного внимания`

Три строки. Всё целиком — это `tril(QKᵀ) V`, и единственное место, где люди
ошибаются, — диагональ.

> Маска **включает диагональ** (`j ≤ t`, а не `j < t`). Соглашение «читаем после
> обновления» говорит, что $o_t$ читает $S_t$, который уже содержит запись самого
> токена $t$. Строгая маска — классический неправильный ответ.

Формы тензоров в области видимости: `q, k` — `(B, H, N, d_k)`; `v` — `(B, H, N, d_v)`;
`causal` — `(N, N)`, где `causal[t, j] = 1` при `j ≤ t`.

In [ ]:
def parallel_form(q, k, v):
    '''o_t = Σ_{j≤t} (q_t · k_j) v_j — квадратичная форма.'''
    n = q.shape[-2]
    causal = torch.ones(n, n, dtype=q.dtype, device=q.device).tril()   # j ≤ t

    # ВАШ КОД — три строки
    # 1. `scores`  (B, H, N, N): каждый запрос против каждого ключа, q kᵀ.
    # 2. `masked`  (B, H, N, N): scores с применённой каузальной маской.
    # 3. `o`       (B, H, N, d_v): замаскированные scores, умноженные на v.
    #
    # В таком виде эти заглушки запускаются и дают неверный ответ — именно так
    # следующая ячейка сообщает вам, что вы ещё не закончили.
    scores = torch.zeros(q.shape[0], q.shape[1], n, n, dtype=q.dtype, device=q.device)
    masked = scores
    o = torch.zeros_like(v)
    # конец
    return o

In [ ]:
# согласуется ли с рекуррентностью?
N1 = 2048
i1 = inputs(N1)
reference = recurrent_linear_attention(i1.q, i1.k, i1.v)   # эталон
diff, atol, BLANK1_OK = residual(parallel_form(i1.q, i1.k, i1.v), reference, N1)
print(f"N={N1}:  max|Δ| = {diff:.3e}   atol = {atol:.3e}")
print("PASS — два взгляда — одна функция" if BLANK1_OK else
      "ЕЩЁ НЕТ — три строки-заглушки всё ещё на месте")

In [ ]:
# эталонный ответ
if not BLANK1_OK:
    print('''
scores = q @ k.transpose(-1, -2)
masked = scores * causal
o      = masked @ v
''')
    print("используем linattn.parallel_linear_attention, чтобы остаток шага выполнился\n")
    parallel_form = parallel_linear_attention

assert_close(parallel_form(i1.q, i1.k, i1.v), reference, n=N1,
             what=f"параллельная форма против рекуррентности при N={N1}")

#### `А теперь — время выполнения`

Рекуррентная форма делает **меньше** арифметики: $O(Nd_kd_v)$ против $O(N^2d_k)$.
Предскажите, какая быстрее, прежде чем запускать следующую ячейку.

Прежде чем что-либо запускать, мы *прикидываем цену*. Наивная параллельная форма
держит два тензора `(B,H,N,N)` одновременно — оценки и замаскированную копию. При
`B·H = 8`, `N = 16384`, fp32 это 17 GiB: OOM даже на **T4**, а не только на
маленькой карте. Поэтому каждая точка сначала сверяется с бюджетом памяти
устройства, и точка, которая не помещается, становится зафиксированным пробелом,
а не падением.

In [ ]:
# что сделала бы T4 в АУДИТОРИИ? (пробный прогон: ничего не выполняется)
LENGTHS = (512, 1024, 2048, 4096, 8192, 16384)
plan_keys = [bench.RunKey(form=f, rule="linear", b=B, h=H, n=n, d_k=D_K, d_v=D_V,
                          chunk_size=CHUNK, dtype="float32", device=str(DEVICE))
             for n in LENGTHS for f in ("parallel", "chunkwise")]
plan = bench.run_sweep(plan_keys, axis="n", dry_run=True,
                       budget_bytes=bench.T4_TOTAL_BYTES)
print(plan.format_table())

In [ ]:
# измеренный перебор (около 20 s)
# `recurrent` останавливается на 8192: это Python-цикл по N, и там он стоит 0.83 s
# за вызов.  Пяти точек уже достаточно, чтобы линия была прямой.
RECURRENT_LENGTHS = (512, 1024, 2048, 4096, 8192)

# Освобождаем карту: наивная параллельная форма при N=8192 хочет 4.3 GiB.
for _name in ("i1", "reference"):
    globals().pop(_name, None)
torch.cuda.empty_cache()

def key_for(form, n):
    return bench.RunKey(form=form, rule="linear", b=B, h=H, n=n, d_k=D_K, d_v=D_V,
                        chunk_size=CHUNK, dtype="float32", device=str(DEVICE), seed=SEED)

keys = []
for n in LENGTHS:
    keys += [key_for(f, n) for f in ("parallel", "chunkwise", "softmax")]
    if n in RECURRENT_LENGTHS:
        keys.append(key_for("recurrent", n))

t0 = time.perf_counter()
sweep = bench.run_sweep(keys, axis="n", warmup=2, iters=3, reduce="min")
print(f"[{time.perf_counter() - t0:.1f} s]")
print(sweep.format_table())

In [ ]:
plotting.plot_sweep(sweep, title="три взгляда на одну функцию, плюс softmax с flash-бэкендом")

#### `Как читать этот график`

**Левая панель — время выполнения.** Не смотрите на две самые левые точки: при
`N ≤ 1024` каждая форма здесь *упирается в запуск ядер* — GPU дольше получает
работу, чем делает её, — а на общем или ноутбучном GPU этот край оси — по большей
части управление частотой. Линии начинают что-то значить с `N = 2048`.

Дальше: у рекуррентной формы *меньше всего* FLOP, и она самая медленная — примерно
в 20× медленнее квадратичной и более чем в 100× медленнее чанковой. Она упирается
в память с арифметической интенсивностью ≈ 1, выполняет `N` последовательных
шагов, а тензорные ядра простаивают.
Квадратичная форма быстра ровно до того момента, когда перестаёт помещаться в
память. Ни одна из них не пригодна для обучения — по двум разным причинам. Это и
чинит шаг 2.

**Правая панель — и вот она переворачивает лекционную картину.** Пожиратель памяти
на этом графике — **наше собственное наивное линейное внимание**, а не
softmax-внимание. Softmax-ядро с flash-бэкендом тоже никогда не материализует
оценки `N×N` — оно `O(N)` по памяти, и вы видите его на тех же осях: оно идёт
вровень с чанковой формой.

Итак: *выигрыш линейного внимания по памяти — не здесь.*

Что остаётся верным — и это на **левой** панели: та же softmax-линия `O(N)` по
памяти и **по-прежнему `O(N²)` по вычислениям** — это единственная линия, которая
загибается вверх, и к `N = 16384` она стоит больше чанковой линейной формы более
чем на порядок. Память `O(N)` даёт возможность *работать* на длинном контексте;
арифметику она не возвращает.

А выигрыш по памяти, который у линейного внимания действительно есть, — в
**состоянии на декодировании**, где это `O(N)` против `O(1)`. Это шаг 6.

### `Шаг 2. Чанковая форма: и то и другое сразу`

*Нужна ячейка 1.*

Разрежьте последовательность на чанки по `L` токенов. **Внутри** чанка используйте
параллельную форму; **между** чанками — рекуррентность, перенося одно состояние
`(B,H,d_k,d_v)` через `⌈N/L⌉` границ. При $Q,K\in\mathbb{R}^{L\times d_k}$,
$V\in\mathbb{R}^{L\times d_v}$, входящем состоянии $S$ и
$M=\mathrm{tril}(\mathbf{1}_{L\times L})$ **с включённой диагональю**:

$$O = QS + \bigl(M\odot QK^{\top}\bigr)V \qquad\qquad S_{\text{out}} = S + K^{\top}V$$

Два слагаемых в выходе: вклад **прошлого** ($QS$, прочитанный *до* того, как записи
самого этого чанка попадут в состояние, — маска их уже учитывает) и вклад **этого
чанка**. Сложность $O(NLd + Nd^2)$: FLOP растут с `L`, число последовательных
шагов падает как `N/L`. `L=1` — рекуррентность, `L≥N` — параллельная форма. Одно
семейство, одна ручка.

#### `Задание 2. Перенос между чанками`

Три строки, внутри цикла. Напишите сами, а не читайте.

Всё вокруг дано: чанкование, дополнение нулями неполного хвоста,
замаскированная матрица оценок и сборка обратно. Не хватает двух слагаемых выхода
и одной строки, которая вносит записи этого чанка в состояние, — и **порядка**, в
котором это происходит, а он и есть всё содержание post-update-соглашения.

In [ ]:
def to_chunks(x, n_chunks, length):
    '''(B,H,N,d) → (B,H,C,L,d), с дополнением оси времени нулями до C·L.

    Дополняющие токены несут q = k = v = 0, поэтому ничего не пишут в состояние и
    не вносят вклада ни в один выход.
    '''
    pad = n_chunks * length - x.shape[2]
    if pad:
        x = F.pad(x, (0, 0, 0, pad))
    return x.reshape(x.shape[0], x.shape[1], n_chunks, length, x.shape[-1])


def chunkwise_form(q, k, v, chunk_size):
    '''Чанковая форма, линейное правило.  Возвращает (o, S_N).'''
    b, h, n, d_k = q.shape
    d_v, L = v.shape[-1], chunk_size
    n_chunks = -(-n // L)                                    # ceil

    qc = to_chunks(q, n_chunks, L)                           # (B,H,C,L,d_k)
    kc = to_chunks(k, n_chunks, L)
    vc = to_chunks(v, n_chunks, L)                           # (B,H,C,L,d_v)
    causal = torch.ones(L, L, dtype=q.dtype, device=q.device).tril()
    scores = (qc @ kc.transpose(-1, -2)) * causal            # (B,H,C,L,L), с маской

    S = torch.zeros(b, h, d_k, d_v, dtype=q.dtype, device=q.device)
    outs = []
    for c in range(n_chunks):
        # ВАШ КОД — три строки
        # В области видимости: qc[:,:,c] (B,H,L,d_k)   kc[:,:,c] (B,H,L,d_k)
        #           vc[:,:,c] (B,H,L,d_v)   scores[:,:,c] (B,H,L,L), с маской
        #           S (B,H,d_k,d_v), состояние, переносимое В этот чанк
        #
        # 1. `inter`: вклад прошлого в выходы этого чанка, Q S.
        # 2. `intra`: вклад этого чанка в самого себя, (M ⊙ QKᵀ) V.
        # 3. `S`:     внести записи этого чанка, S ← S + Kᵀ V.  Порядок важен:
        #             выходы выше читают S ДО того, как выполнится эта строка.
        inter = torch.zeros(b, h, L, d_v, dtype=q.dtype, device=q.device)   # TODO 1
        intra = torch.zeros_like(inter)                                     # TODO 2
        S = S + 0.0                                                         # TODO 3
        # конец
        outs.append(inter + intra)

    o = torch.stack(outs, dim=2).reshape(b, h, n_chunks * L, d_v)
    return o[:, :, :n], S                                    # отбросить дополненный хвост

In [ ]:
# каждый размер чанка, включая те, что не делят N
# «Совпадает для каждого L» надо показывать на длинах, которые НЕ кратны L.
N2 = 1000                       # не кратно ничему из того, что мы сейчас используем
CHUNK_SIZES = (1, 7, 16, 64, 128, N2, N2 + 13)     # L=1, неполные хвосты, L=N, L>N
i2 = inputs(N2)
ref_par = parallel_linear_attention(i2.q, i2.k, i2.v)
ref_rec, ref_state = recurrent_linear_attention(i2.q, i2.k, i2.v, return_state=True)

rows, BLANK2_OK = [], True
for L in CHUNK_SIZES:
    o, S = chunkwise_form(i2.q, i2.k, i2.v, L)
    d_o, atol_o, ok_o = residual(o, ref_par, N2)
    d_s, atol_s, ok_s = residual(S, ref_state, N2)
    BLANK2_OK &= ok_o and ok_s
    tail = "—" if N2 % L == 0 else str(N2 % L)
    rows.append(f"  L={L:<6d} хвост={tail:<5s} max|Δo| = {d_o:.2e}  max|ΔS| = {d_s:.2e}"
                f"   {'ok' if ok_o and ok_s else 'НЕТ'}")
print(f"N={N2}, допуск atol = {atol_o:.2e} (выходы) / {atol_s:.2e} (состояние)")
print("\n".join(rows))
print("\nPASS — одна функция для каждого размера чанка" if BLANK2_OK else
      "\nЕЩЁ НЕТ — три строки-заглушки всё ещё на месте")

In [ ]:
# эталонный ответ
if not BLANK2_OK:
    print('''
inter = qc[:, :, c] @ S
intra = scores[:, :, c] @ vc[:, :, c]
S     = S + kc[:, :, c].transpose(-1, -2) @ vc[:, :, c]
''')
    print("используем linattn.chunkwise_linear_attention, чтобы остаток шага выполнился\n")
    chunkwise_form = lambda q, k, v, L: chunkwise_linear_attention(
        q, k, v, chunk_size=L, return_state=True)

for L in CHUNK_SIZES:
    o, S = chunkwise_form(i2.q, i2.k, i2.v, L)
    assert_close(o, ref_par, n=N2, what=f"chunkwise(L={L}) против параллельной формы")
    assert_close(o, ref_rec, n=N2, what=f"chunkwise(L={L}) против рекуррентности")
    assert_close(S, ref_state, n=N2, what=f"chunkwise(L={L}) конечное состояние")

In [ ]:
# библиотечная версия, все три правила, выходы и состояние
# С гейтом: S_t = α_t S_{t-1} + k_t v_tᵀ   (затухание ДО записи)
# Дельта:   S_t = (I − β_t k_t k_tᵀ) S_{t-1} + β_t k_t v_tᵀ   (стирающий множитель СЛЕВА)
# Совмещённое правило «гейт + дельта» — только лекционный материал: флага для него
# в этой библиотеке нет, так что не ищите.
for rule in linattn.RULES:
    ir = inputs(N2, rule=rule)
    kwargs = ir.rule_kwargs()
    o_ref, s_ref = recurrent_linear_attention(ir.q, ir.k, ir.v, return_state=True, **kwargs)
    o_chk, s_chk = chunkwise_linear_attention(ir.q, ir.k, ir.v, chunk_size=CHUNK,
                                              return_state=True, **kwargs)
    assert_close(o_chk, o_ref, n=N2, what=f"{rule:<7s} chunkwise, выходы")
    assert_close(s_chk, s_ref, n=N2, what=f"{rule:<7s} chunkwise, состояние")

#### `U-кривая по L`

FLOP растут с `L`; число последовательных шагов падает как `N/L`. Оптимум
удовлетворяет $L^{\ast}\propto\sqrt{\tau F}$, где `F` — достигнутые FLOP/s, а `τ` —
накладные расходы на чанк. В реализации на PyTorch `τ` определяется в основном
**стоимостью Python-цикла и запуска ядер**, поэтому измеренный минимум лежит
*выше*, чем предсказывает чисто FLOP-ный аргумент, и **сдвигается вместе с CPU
хоста**.

Предсказывайте механизм и диапазон, а не число. На карте разработки дно широкое и
лежит около `L ∈ [128, 256]`. У вас может быть иначе; что не должно отличаться —
это форма кривой.

In [ ]:
torch.cuda.empty_cache()
t0 = time.perf_counter()
ucurve = bench.sweep_chunk_size(chunk_sizes=(1, 8, 16, 32, 64, 128, 256, 512, 1024),
                                n=8192, b=B, h=H, d_k=D_K, d_v=D_V,
                                device=DEVICE, warmup=2, iters=5, reduce="min")
print(f"[{time.perf_counter() - t0:.1f} s]")
best = min(ucurve.ok, key=lambda p: p.seconds)
for p in ucurve.ok:
    mark = "  <-- минимум" if p is best else ""
    print(f"  L={p.key.chunk_size:<5d} {p.seconds * 1e3:8.2f} ms{mark}")
print(f"\nL=1 — это рекуррентность: {ucurve.ok[0].seconds / best.seconds:.0f}× от оптимума, "
      f"тот же путь в коде, изменён один аргумент.")

In [ ]:
plotting.plot_times(ucurve)

### `Шаг 3. Те же четыре строки, на Triton`

*Нужна ячейка 1.*

Чанковая форма на PyTorch запускает горстку CUDA-ядер **на каждый чанк** и между
ними гоняет состояние туда-обратно в HBM. Triton-ядро делает всю
последовательность за один запуск, и `S` живёт в регистрах от первого токена до
последнего. В этом вся идея: *состояние — это то, что вы держите в SRAM, и оно
размером `d_k × d_v`, какой бы длинной ни была последовательность.*

Всё ниже дано вам, кроме **трёх строк**: арифметика указателей, маски, неполный
последний чанк, конфигурация запуска и проверки области допустимых входов уже на
месте. Вы пишете саму чанковую форму — те же $O = QS + (M\odot QK^{\top})V$,
$S \leftarrow S + K^{\top}V$, которые вы написали в шаге 2.

Две вещи в арифметике, которые уже сделаны правильно и в которых легко ошибиться:

* каждый `tl.dot` передаёт `input_precision="ieee"`. Без этого Triton выдаёт TF32
  `mma.sync` на Ampere и обычный fp32 на T4 в аудитории, и две машины расходятся
  численно.
* fp32-произведение **не** выполняется на тензорных ядрах на `sm_75`. Ускорение
  берётся из тайлинга и из того, что `S` остаётся в регистрах, а не из тензорных
  ядер.

In [ ]:
# упражнение из linattn.seminar
from linattn.seminar import blank_region, skeleton_source, solution_source
print(blank_region(skeleton_source()))

#### `Задание 3. Ядро`

Отредактируйте следующую ячейку (ищите `BEGIN STUDENT BLANK`) и запустите её. В
исходном виде три заглушки **компилируются и работают**; они просто считают не
то, поэтому ячейка-проверка после неё сообщает вам число, а не трейсбек.

In [ ]:
"""Triton-шаг семинара — половина skeleton.

Упражнение — ровно то, что стоит между маркерами ``BEGIN STUDENT BLANK`` и
``END STUDENT BLANK``: четыре оператора, которые *и есть* чанковая форма.  Всё
остальное — арифметика указателей, маски, неполный последний чанк, запуск,
проверки области допустимых входов — дано.

Что ядро вычисляет для одной пары ``(batch, head)``, чанк за чанком, где
``M = tril(ones(L, L))`` — маска **с включённой диагональю** (соглашение «читаем
после обновления»: ``o_t`` читает состояние, уже содержащее токен ``t``)::

    O = Q S + (M . (Q K^T)) V              S_out = S + K^T V

Только прямой проход, только линейное правило.  Всё обучение идёт через
:func:`linattn.chunkwise.chunkwise_linear_attention` посредством autograd.

Две вещи в арифметике, в которых легко ошибиться и которые здесь уже сделаны
правильно:

* каждый ``tl.dot`` фиксирует ``input_precision="ieee"``.  Без фиксации Triton
  выдаёт TF32 ``mma.sync`` на карте разработки (``sm_86``) и обычный fp32 на T4 в
  аудитории (``sm_75``), так что две машины расходились бы численно.
* fp32-произведение **не** выполняется на тензорных ядрах на ``sm_75``.  Ускорение
  берётся из тайлинга и из того, что ``S`` держится в регистрах всю
  последовательность.
"""

from __future__ import annotations

import triton
import triton.language as tl
from torch import Tensor

from linattn.kernel import DEFAULT_KERNEL_CHUNK_SIZE, launch_chunkwise_kernel

__all__ = [
    "skeleton_chunkwise_linear_attention",
    "skeleton_chunkwise_linear_attention_fwd",
]


@triton.jit
def skeleton_chunkwise_linear_attention_fwd(
    q_ptr,  # (B, H, N, d_k), непрерывный
    k_ptr,  # (B, H, N, d_k), непрерывный
    v_ptr,  # (B, H, N, d_v), непрерывный
    o_ptr,  # (B, H, N, d_v), непрерывный — записывается
    state_ptr,  # (B, H, d_k, d_v), непрерывный — читается И записывается
    N,
    BLOCK_L: tl.constexpr,  # размер чанка L
    BLOCK_DK: tl.constexpr,  # d_k
    BLOCK_DV: tl.constexpr,  # d_v
):
    """Одна программа на пару ``(batch, head)``; чанки обходятся по порядку.

    Каждый тензор непрерывен, поэтому срез ``(b, h)`` тензора ``(B,H,N,d)``
    начинается с ``pid * N * d``, а его элемент ``(t, i)`` лежит по смещению
    ``t * d + i``.  Это вся арифметика указателей в ядре.
    """
    pid = tl.program_id(0)  # = b * H + h

    q_base = q_ptr + pid * N * BLOCK_DK
    k_base = k_ptr + pid * N * BLOCK_DK
    v_base = v_ptr + pid * N * BLOCK_DV
    o_base = o_ptr + pid * N * BLOCK_DV

    offs_l = tl.arange(0, BLOCK_L)
    offs_dk = tl.arange(0, BLOCK_DK)
    offs_dv = tl.arange(0, BLOCK_DV)

    # Состояние загружается один раз и живёт в регистрах всю последовательность.
    state_ptrs = (
        state_ptr
        + pid * BLOCK_DK * BLOCK_DV
        + offs_dk[:, None] * BLOCK_DV
        + offs_dv[None, :]
    )
    state = tl.load(state_ptrs)

    # Диагональ включена.  Строгая маска — классический неправильный ответ.
    causal = offs_l[:, None] >= offs_l[None, :]

    for start in range(0, N, BLOCK_L):
        offs_n = start + offs_l
        mask = offs_n < N  # неполный последний чанк

        q = tl.load(
            q_base + offs_n[:, None] * BLOCK_DK + offs_dk[None, :],
            mask=mask[:, None],
            other=0.0,
        )
        k = tl.load(
            k_base + offs_n[:, None] * BLOCK_DK + offs_dk[None, :],
            mask=mask[:, None],
            other=0.0,
        )
        v = tl.load(
            v_base + offs_n[:, None] * BLOCK_DV + offs_dv[None, :],
            mask=mask[:, None],
            other=0.0,
        )

        # --- BEGIN STUDENT BLANK ---
        # Три вещи, в этом порядке.  Формы, для справки:
        #     q (L, d_k)   k (L, d_k)   v (L, d_v)   state (d_k, d_v)
        #     scores (L, L)             o (L, d_v)
        #
        # TODO 1. `scores`: матрица оценок чанка с маской, включающей диагональ,
        #         M . (Q K^T).  `causal` — маска; tl.where её применяет.
        # TODO 2. `o`: выход этого чанка, Q S + scores V.  S — состояние,
        #         перенесённое из предыдущих чанков; его нужно прочитать *до*
        #         того, как в него попадут записи самого этого чанка, — маска
        #         выше их уже учитывает.
        # TODO 3. внести записи этого чанка в состояние: S += K^T V.
        #
        # Транспонируйте через tl.trans(x) и передавайте input_precision="ieee"
        # в каждое произведение, иначе железо разработки и железо в аудитории
        # разойдутся.
        #
        # Замените три заглушки ниже.  В исходном виде они компилируются и
        # работают, а проверка в следующей ячейке падает — так вы и узнаёте,
        # что ещё не закончили.
        scores = tl.zeros([BLOCK_L, BLOCK_L], dtype=tl.float32)  # TODO 1
        o = tl.zeros([BLOCK_L, BLOCK_DV], dtype=tl.float32)  # TODO 2
        state = state + 0.0  # TODO 3
        # --- END STUDENT BLANK ---

        tl.store(
            o_base + offs_n[:, None] * BLOCK_DV + offs_dv[None, :],
            o,
            mask=mask[:, None],
        )

    tl.store(state_ptrs, state)


def skeleton_chunkwise_linear_attention(
    q: Tensor,
    k: Tensor,
    v: Tensor,
    *,
    chunk_size: int = DEFAULT_KERNEL_CHUNK_SIZE,
    initial_state: Tensor | None = None,
    return_state: bool = False,
) -> Tensor | tuple[Tensor, Tensor]:
    """Запустить ядро выше.

    Валидация, выделение памяти и конфигурация запуска — библиотечные, поэтому
    эта функция отказывает ровно в том, в чём отказывает
    :func:`linattn.kernel.triton_chunkwise_linear_attention`, и с теми же
    сообщениями: размер чанка, ``d_k`` или ``d_v`` меньше 16 или не степень
    двойки, dtype, отличный от float32, и тайлинг, превышающий 64 KB разделяемой
    памяти T4.
    """
    return launch_chunkwise_kernel(
        skeleton_chunkwise_linear_attention_fwd,
        q,
        k,
        v,
        chunk_size=chunk_size,
        initial_state=initial_state,
        return_state=return_state,
    )

In [ ]:
# согласуется ли ваше ядро с PyTorch из шага 2?
N3, L3 = 8192, 32          # L=32 при d_k=d_v=64 требует 44 KiB разделяемой памяти;
                           # L=64 потребовал бы 80 KiB, а у T4 их 64. См. шаг 4.
i3 = inputs(N3)
ref_o, ref_S = chunkwise_linear_attention(i3.q, i3.k, i3.v, chunk_size=L3,
                                          return_state=True)
KERNEL = skeleton_chunkwise_linear_attention
got_o, got_S = KERNEL(i3.q, i3.k, i3.v, chunk_size=L3, return_state=True)
d_o, atol_o, ok_o = residual(got_o, ref_o, N3)
d_s, atol_s, ok_s = residual(got_S, ref_S, N3)
BLANK3_OK = ok_o and ok_s
print(f"N={N3}, L={L3}:  max|Δo| = {d_o:.3e} (atol {atol_o:.2e})   "
      f"max|ΔS| = {d_s:.3e} (atol {atol_s:.2e})")
print("PASS — вы написали GPU-ядро" if BLANK3_OK else
      "ЕЩЁ НЕТ — TODO 1/2/3 всё ещё заглушки (оно запустилось; просто считает не то)")

In [ ]:
# эталонный ответ
if not BLANK3_OK:
    print(blank_region(solution_source()))
    print("подключаем linattn.seminar.triton_solution, чтобы остаток шага выполнился\n")
    from linattn.seminar.triton_solution import solution_chunkwise_linear_attention
    KERNEL = solution_chunkwise_linear_attention

o, S = KERNEL(i3.q, i3.k, i3.v, chunk_size=L3, return_state=True)
assert_close(o, ref_o, n=N3, what=f"Triton-ядро против чанковой формы на PyTorch, N={N3} L={L3}")
assert_close(S, ref_S, n=N3, what="конечное состояние Triton-ядра")

In [ ]:
# область допустимых входов ядра
# `tl.dot` требует, чтобы КАЖДАЯ размерность была ≥ 16, и это ограничивает d_k и d_v
# ровно так же, как размер чанка.  Ядро, которое молча считает неверно за пределами
# своей области, хуже ядра, которое отказывается.
def refuses(what, **over):
    n, d_k, d_v, L = over.get("n", 64), over.get("d_k", D_K), over.get("d_v", D_V), over.get("L", 32)
    dtype = over.get("dtype", DTYPE)
    q = torch.zeros(1, 1, n, d_k, device=DEVICE, dtype=dtype)
    v = torch.zeros(1, 1, n, d_v, device=DEVICE, dtype=dtype)
    try:
        linattn.triton_chunkwise_linear_attention(q, q, v, chunk_size=L)
    except linattn.KernelDomainError as error:
        print(f"  PASS  отвергнуто {what}: {str(error).splitlines()[0][:96]}")
        return
    raise AssertionError(f"ядро приняло {what}")

refuses("chunk_size=8  (меньше минимума 16 для tl.dot)", L=8)
refuses("d_k=8         (меньше минимума 16 для tl.dot)", d_k=8)
refuses("d_v=8         (меньше минимума 16 для tl.dot)", d_v=8)
refuses("bfloat16      (запрещён на sm_75)", dtype=torch.bfloat16)
refuses("L=64 при d=64 (80 KiB > 64 KiB разделяемой памяти у T4)", L=64)
print("  PASS  отказы по области допустимых входов")

### `Шаг 4. А быстрее ли?`

*Нужна ячейка 1.*

In [ ]:
from linattn.seminar.triton_solution import solution_chunkwise_linear_attention as tri
torch.cuda.empty_cache()
N4 = 8192
i4 = inputs(N4)

print(f"N={N4}, B={B}, H={H}, d_k=d_v={D_K}, fp32 — ms на один прямой проход "
      f"(лучший из 5, после 2 прогревочных вызовов)\n")
print(f"{'L':>6} {'PyTorch (шаг 2)':>18} {'Triton (шаг 3)':>17} {'shared mem':>12}  влезает в T4")
for L in (16, 32, 64, 128, 256):
    shmem = linattn.kernel_shared_memory_bytes(chunk_size=L, d_k=D_K, d_v=D_V)
    fits = shmem <= linattn.TURING_SHARED_MEMORY_BYTES
    t_pt = bench.time_callable(
        lambda: chunkwise_linear_attention(i4.q, i4.k, i4.v, chunk_size=L),
        device=DEVICE, warmup=2, iters=5, reduce="min").seconds * 1e3
    if fits:
        t_tr = bench.time_callable(
            lambda: tri(i4.q, i4.k, i4.v, chunk_size=L),
            device=DEVICE, warmup=2, iters=5, reduce="min").seconds * 1e3
        tr = f"{t_tr:.2f}  ({t_pt / t_tr:.1f}×)"
    else:
        tr = "отказ"
    print(f"{L:>6} {t_pt:>18.2f} {tr:>17} {shmem // 1024:>9} KiB  {'да' if fits else 'НЕТ'}")

In [ ]:
# ограничение, которое порождает таблицу выше
big = linattn.kernel_shared_memory_bytes(chunk_size=64, d_k=D_K, d_v=D_V)
assert big > linattn.TURING_SHARED_MEMORY_BYTES, "L=64 при d=64 не должно влезать в Turing"
ok = linattn.kernel_shared_memory_bytes(chunk_size=32, d_k=D_K, d_v=D_V)
assert ok <= linattn.TURING_SHARED_MEMORY_BYTES
print(f"  PASS  L=64, d_k=d_v={D_K} требует {big // 1024} KiB > "
      f"{linattn.TURING_SHARED_MEMORY_BYTES // 1024} KiB; L=32 требует {ok // 1024} KiB")
print(f"        один только тайл состояния — d_k·d_v·4 = {D_K * D_V * 4 // 1024} KiB из них")
if DEVICE.type == "cuda":
    print(f"        сетка = B·H = {B * H} программ на "
          f"{torch.cuda.get_device_properties(0).multi_processor_count} SM")

#### `Что таблица говорит, а чего — нет`

**При одном и том же размере чанка ядро выигрывает** — в несколько раз; прочитайте
это в таблице, которую вы только что получили. Этот разрыв *сокращается* с ростом
`L`, потому что стоимость PyTorch на чанк — это итерация Python плюс запуск ядра
— та самая `τ`, что подняла минимум U-кривой в шаге 2, — а у Triton-ядра **один**
запуск, каким бы ни был `L`.

**Но PyTorch может выбрать большой `L`, а ядро — нет.** Посмотрите на последний
столбец. При `d_k = d_v = 64` тайлингу нужно
`4·(2Ld_k + Ld_v + L² + d_kd_v)` байт разделяемой памяти, и при `L = 64` это уже
80 KiB против 64 KiB у Turing. Всё, что выше, ядро отвергает. Так что сравните
два *лучших* результата в таблице, которую вы только что получили, — PyTorch на
своём лучшем `L` против ядра на его лучшем **допустимом** — и преимущество ядра
на чанк оказалось потрачено на ограничение, которое никак не связано с тем,
насколько ядро хорошо.

Отсюда следует:

* **тайл состояния — это `d_k·d_v·4` байт, и он конкурирует за ту же SRAM, что и
  всё остальное** — при `d_k = d_v = 128` это 64 KiB, весь бюджет Turing, и это
  аппаратная формулировка тезиса этого модуля;
* сетка — это `B·H` программ, так что при `B = 1` этому ядру не хватает занятости
  (occupancy);
* fp32-произведение **не** идёт на тензорных ядрах на `sm_75`;
* и потому *«моё ядро быстрее PyTorch»* — утверждение, зависящее от железа, и
  ничто в этом модуле его не проверяет. Что мы проверяем — что ядро
  **корректно** (шаг 3), и именно эта часть переносится дальше.

### `Шаг 5. Настоящий слой: читаем Qwen3-Next`

*Нужна ячейка 1.*

Qwen3-Next — выпущенный гибрид: 48 слоёв, три слоя линейного внимания на каждый
слой полного внимания. Его линейные слои — **gated DeltaNet**. Файл уже на этой
машине, внутри `transformers`. Мы прочитаем из него рекуррентность, строка за
строкой, а затем *докажем*, что это та же функция, против которой вы всё утро
писали проверки.

Читаем мы при этом **референсную реализацию**, а не то, что считает на проде.
`transformers` — это место, где живёт *определение* модели; исполняют его vLLM и
SGLang, а сам gated DeltaNet в Qwen3-Next идёт через чанковые ядра из
`flash-linear-attention`, и без этого пакета слой молча откатывается ровно на те
функции `torch_*_gated_delta_rule`, которые мы сейчас откроем. Это и делает их
удобными: они написаны затем, чтобы задавать эталон численности.

In [ ]:
import inspect
from transformers import Qwen3NextConfig
from transformers.models import qwen3_next
from transformers.models.qwen3_next.modeling_qwen3_next import (
    torch_chunk_gated_delta_rule, torch_recurrent_gated_delta_rule)

# Файл, в котором HF определяет модель.  Он поставляется внутри пакета, так что
# это локальное чтение: семинару для этого не нужна сеть.
modular = Path(qwen3_next.__file__).with_name("modular_qwen3_next.py")
if modular.exists():
    print(modular.as_posix())
    lines = modular.read_text(encoding="utf-8").splitlines()
    offset = 0
else:                                    # откатываемся на сгенерированный модуль
    print(inspect.getsourcefile(torch_recurrent_gated_delta_rule))
    lines = inspect.getsource(torch_recurrent_gated_delta_rule).splitlines()
    offset = inspect.getsourcelines(torch_recurrent_gated_delta_rule)[1] - 1
start = next(n for n, line in enumerate(lines) if "for i in range(sequence_length)" in line)
print(f"\n  torch_recurrent_gated_delta_rule — весь слой, один раз на токен:\n")
for n, line in enumerate(lines[start:start + 12], start=start + 1 + offset):
    print(f"{n:>5} {line}")

Сопоставьте с тем, что на доске, строка за строкой:

| их строка | наши обозначения |
|---|---|
| `last_recurrent_state = last_recurrent_state * g_t` | $\alpha_t S_{t-1}$ — **затухание до записи** |
| `kv_mem = (state * k_t).sum(-2)` | $S_{t-1}^{\top}k_t$ — что состояние сейчас возвращает для этого ключа |
| `delta = (v_t - kv_mem) * beta_t` | $w_t=\beta_t(v_t-S_{t-1}^{\top}k_t)$ — ошибка, умноженная на шаг обучения |
| `state = state + k_t ⊗ delta` | $S_t=S_{t-1}+k_tw_t^{\top}$ — дельта-правило, стирающий множитель **слева**, записанное как обновление ранга один |
| `out[i] = (state * q_t).sum(-2)` | $o_t=S_t^{\top}q_t$ — **post-update**: читается состояние, которое только что записали |

Два отличия, на которые нужно сделать поправку, и оба — суть дела:

1. **Они масштабируют запрос на $1/\sqrt{d_k}$; мы намеренно этого не делаем.**
   Лекция выводит линейное внимание *из* нормированного softmax-внимания, а эта
   библиотека отбрасывает и $1/\sqrt d$, и знаменатель $\sum\phi(k)$. Поэтому для
   сравнения передайте нашей версии `q/√d_k`.
2. **У них — gated *delta*; в этой библиотеке гейт и дельта есть по отдельности.**
   При `g = 0` получаем $\alpha = 1$, и их правило схлопывается в наше чистое
   дельта-правило. Совмещённое правило — лекционный материал: **флага для него в
   `linattn` нет, так что не ищите.**

In [ ]:
# linattn против референсной реализации Qwen3-Next
torch.manual_seed(SEED)
b5, h5, n5, d5 = 2, 4, 64, 32
q5 = torch.randn(b5, h5, n5, d5)
k5 = torch.randn(b5, h5, n5, d5)
k5 = k5 / k5.norm(dim=-1, keepdim=True)         # ключи дельта-правила L2-нормированы
v5 = torch.randn(b5, h5, n5, d5)
beta5 = 0.2 + 0.75 * torch.rand(b5, h5, n5)     # в самой модели β — сигмоида: β < 1
g5 = torch.zeros(b5, h5, n5)                    # log α = 0  ⇒  α = 1  ⇒  чистая дельта

o_hf, s_hf = torch_recurrent_gated_delta_rule(         # их раскладка — (B, N, H, d)
    query=q5.transpose(1, 2).contiguous(), key=k5.transpose(1, 2).contiguous(),
    value=v5.transpose(1, 2).contiguous(), g=g5.transpose(1, 2).contiguous(),
    beta=beta5.transpose(1, 2).contiguous(),
    initial_state=None, output_final_state=True, use_qk_l2norm_in_kernel=False)
o_hf = o_hf.transpose(1, 2)

o_rec, s_rec = decode(q5 * d5 ** -0.5, k5, v5, None, rule="delta", beta=beta5)
o_chk, s_chk = chunkwise_linear_attention(q5 * d5 ** -0.5, k5, v5, chunk_size=16,
                                          rule="delta", beta=beta5, return_state=True)
assert_close(o_rec, o_hf, n=n5, what="linattn.decode      == рекуррентность Qwen3-Next")
assert_close(s_rec, s_hf, n=n5, what="…и конечное состояние, которое она оставляет")
assert_close(o_chk, o_hf, n=n5, what="linattn.chunkwise   == рекуррентность Qwen3-Next")

In [ ]:
# два числа, посчитанные из поставляемого конфига
cfg = Qwen3NextConfig()
kinds = {t: cfg.layer_types.count(t) for t in sorted(set(cfg.layer_types))}
print(f"{cfg.num_hidden_layers} слоёв: {kinds}  → "
      f"гибрид {kinds['linear_attention'] / kinds['full_attention']:.0f}:1\n")

FP16 = 2
lin  = bench.LinearAttentionLayer(heads=cfg.linear_num_value_heads,
                                  d_k=cfg.linear_key_head_dim,
                                  d_v=cfg.linear_value_head_dim, dtype_bytes=FP16)
full = bench.SoftmaxAttentionLayer(n_kv_heads=cfg.num_key_value_heads,
                                   head_dim=cfg.head_dim, dtype_bytes=FP16)
N_ctx = 131072
print(f"байты состояния на слой при N={N_ctx} и прирост байтов на токен, на слой:")
print(f"  линейный слой : {bench.format_bytes(lin.state_bytes(N_ctx)):>10}   "
      f"прирост {bench.format_bytes(lin.marginal_bytes_per_token()):>10} / токен")
print(f"  полный слой   : {bench.format_bytes(full.state_bytes(N_ctx)):>10}   "
      f"прирост {bench.format_bytes(full.marginal_bytes_per_token()):>10} / токен\n")

hybrid_marginal = kinds["full_attention"] * full.marginal_bytes_per_token()
all_full = cfg.num_hidden_layers * full.marginal_bytes_per_token()
print(f"прирост по всей модели, гибрид     : {bench.format_bytes(hybrid_marginal)} / токен")
print(f"прирост по всей модели, всё полное : {bench.format_bytes(all_full)} / токен")
print(f"соотношение: {hybrid_marginal / all_full:.0%}  ← то самое «сокращение KV-кэша на 75%»")

assert lin.marginal_bytes_per_token() == 0
assert hybrid_marginal * 4 == all_full, "прирост гибрида 3:1 должен быть ровно 25%"
print("\n  PASS  прирост гибрида — ровно 25% от прироста полного внимания")

Эти 25% — утверждение о **приросте** — байтах на *дополнительный* токен — и ни о
чём больше. Три линейных слоя из каждых четырёх по-прежнему держат состояние на
слой, и это состояние вообще не меняется с `N`. Приводить одно число под ярлыком
вроде «байты состояния на токен» — значит смешивать две разные величины с
разными единицами, и именно так это сравнение перестаёт сходиться.

### `Шаг 6. Декодирование выходит на плато`

*Нужна ячейка 1.*

У генерации два режима. **Префилл** видит весь промпт и хочет чанковую форму.
**Декодирование** видит по одному токену, ему нечего батчить, и это
рекуррентность — одно обновление ранга один, `O(d_k d_v)` работы, **независимо
от того, сколько токенов было до этого**.

Префилл уже держит конечное состояние, когда его цикл по чанкам заканчивается,
поэтому он передаёт его дальше, а не выводит заново. (Заново выводить состояние
32k-токенного промпта Python-циклом по токенам — это 32768 итераций
интерпретатора на каждую измеренную точку, и это обесценивает весь смысл
чанковой формы.)

Сначала докажем, что префилл с последующим декодированием — та же функция, что
и один чанковый проход.

In [ ]:
# префилл + step == один чанковый проход
for N6 in (1000, 4096):          # 1000 не кратно размеру чанка
    K6 = 8
    i6 = inputs(N6 + K6)
    whole = chunkwise_linear_attention(i6.q, i6.k, i6.v, chunk_size=CHUNK)
    o_prompt, S = prefill(i6.q[:, :, :N6], i6.k[:, :, :N6], i6.v[:, :, :N6],
                          chunk_size=CHUNK)
    pieces = [o_prompt]
    for t in range(N6, N6 + K6):
        o_t, S = decode_step(i6.q[:, :, t:t + 1], i6.k[:, :, t:t + 1],
                             i6.v[:, :, t:t + 1], S)
        pieces.append(o_t)
    assert_close(torch.cat(pieces, dim=2), whole, n=N6 + K6,
                 what=f"prefill({N6}) + {K6}×step == chunkwise({N6 + K6})")
    print(f"        состояние, перенесённое через стык: {tuple(S.shape)}, "
          f"{bench.format_bytes(S.numel() * S.element_size())}")

In [ ]:
# один шаг декодирования против промпта, который всё удлиняется
PROMPTS = (1024, 4096, 16384, 65536, 131072)
lin_ms, sm_ms, kv_bytes = [], [], []
t0 = time.perf_counter()
for N6 in PROMPTS:
    i6 = inputs(N6)
    o_pref, S = prefill(i6.q, i6.k, i6.v, chunk_size=CHUNK)   # ← S берётся отсюда
    del o_pref                                                #   и никогда не выводится заново
    nxt = inputs(1, seed=SEED + 1)

    # линейное внимание: одно обновление ранга один против состояния, которое не растёт
    lin_ms.append(bench.time_callable(
        lambda: decode_step(nxt.q, nxt.k, nxt.v, S),
        device=DEVICE, warmup=5, iters=20, reduce="min").seconds * 1e3)

    # softmax-внимание: один запрос против всех ключей и значений, что когда-либо были
    k_cache = torch.cat([i6.k, nxt.k], dim=2)
    v_cache = torch.cat([i6.v, nxt.v], dim=2)
    sm_ms.append(bench.time_callable(
        lambda: F.scaled_dot_product_attention(nxt.q, k_cache, v_cache),
        device=DEVICE, warmup=5, iters=20, reduce="min").seconds * 1e3)

    kv_bytes.append(k_cache.numel() * 4 + v_cache.numel() * 4)
    print(f"N={N6:>7}   шаг linear {lin_ms[-1]:6.3f} ms   шаг softmax {sm_ms[-1]:7.3f} ms"
          f"   KV-кэш {bench.format_bytes(kv_bytes[-1]):>9}"
          f"   состояние {bench.format_bytes(S.numel() * 4):>9}")
    del i6, S, nxt, k_cache, v_cache
    torch.cuda.empty_cache()
print(f"[{time.perf_counter() - t0:.1f} s]  "
      f"linear: {min(lin_ms):.3f}–{max(lin_ms):.3f} ms на диапазоне {PROMPTS[-1] // PROMPTS[0]}× "
      f"по N.  softmax: {max(sm_ms) / min(sm_ms):.0f}× от края до края.")

In [ ]:
import matplotlib.pyplot as plt
S_BYTES = H * D_K * D_V * 4
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(PROMPTS, lin_ms, marker="o", label="линейное внимание: step()")
axes[0].plot(PROMPTS, sm_ms, marker="o", label="softmax-внимание: запрос к KV-кэшу")
axes[0].set_xscale("log", base=2); axes[0].set_yscale("log")
axes[0].set_xlabel("длина уже декодированного промпта N"); axes[0].set_ylabel("один шаг декодирования (ms)")
axes[0].set_title("время на один декодированный токен", fontsize="medium")
axes[0].grid(True, which="both", alpha=0.3); axes[0].legend(frameon=False, fontsize="small")

axes[1].plot(PROMPTS, [S_BYTES] * len(PROMPTS), marker="o",
             label=f"линейное состояние: H·d_k·d_v·4 = {bench.format_bytes(S_BYTES)}, прирост 0 B/токен")
axes[1].plot(PROMPTS, kv_bytes, marker="o",
             label=f"KV-кэш: прирост {bench.format_bytes(2 * H * D_K * 4)}/токен")
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log")
axes[1].set_xlabel("длина промпта N"); axes[1].set_ylabel("байты, переносимые к следующему токену")
axes[1].set_title("что приходится хранить", fontsize="medium")
axes[1].grid(True, which="both", alpha=0.3); axes[1].legend(frameon=False, fontsize="small")
fig.suptitle(f"декодирование: B={B}, H={H}, d_k={D_K}, d_v={D_V}, fp32, чанк L={CHUNK}",
             fontsize="medium")
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig

In [ ]:
# два числа модуля, для этой конфигурации
ours = bench.LinearAttentionLayer(heads=H, d_k=D_K, d_v=D_V, dtype_bytes=4)
theirs = bench.SoftmaxAttentionLayer(n_kv_heads=H, head_dim=D_K, dtype_bytes=4)
for n in (4096, 32768, 131072):
    print(f"N={n:>7}   состояние linear {bench.format_bytes(ours.state_bytes(n)):>9}"
          f"   состояние softmax {bench.format_bytes(theirs.state_bytes(n)):>9}"
          f"   соотношение {theirs.state_bytes(n) / ours.state_bytes(n):8.0f}×")
assert ours.marginal_bytes_per_token() == 0
assert len({ours.state_bytes(n) for n in (1, 4096, 131072, 10 ** 9)}) == 1, \
    "линейное состояние должно быть одного размера при любой длине контекста"
assert theirs.marginal_bytes_per_token() == 2 * H * D_K * 4 != 0
print(f"\n  PASS  linear: состояние на слой {bench.format_bytes(ours.state_bytes(0))} при ЛЮБОМ N, "
      f"прирост 0 B/токен")
print(f"  PASS  softmax: прирост {bench.format_bytes(theirs.marginal_bytes_per_token())}"
      f"/токен, постоянный и ненулевой")

In [ ]:
plotting.plot_state_bytes()      # референсная модель модуля: в форме Llama-3-8B

#### `Что показывают эти панели`

Левая панель **плоская на 128×-диапазоне длины контекста**, а другая линия —
подъём на два порядка. Подъём — это ровно KV-кэш, который перечитывается: каждый
декодированный токен платит за каждый токен до него.

Прочитайте пересечение честно. На коротком контексте softmax-декодирование
*дешевле* — внимание по 1024 ключам — это одно маленькое ядро, а наш `step()`
платит фиксированные несколько сотен микросекунд накладных расходов фреймворка за
вызов при любом `N`. Эта фиксированная цена **и есть** плоская линия, и в этом
весь смысл: она не двигается. Где-то в районе нескольких тысяч токенов подъём
пересекает её, и после этого разрыв только растёт.

Правая панель — то же утверждение в байтах. Оба числа модуля живут там, и это
*разные величины с разными единицами*:

* **байты состояния на слой при длине контекста `N`** — `H·d_k·d_v·dtype_bytes`
  для линейного слоя, одно и то же при любом `N`; `2·n_kv_heads·head_dim·dtype_bytes·N`
  для softmax, линейно по `N`.
* **прирост байтов на токен, на слой** — **ровно ноль** для любой архитектуры с
  линейным вниманием; постоянный и ненулевой для softmax.

Формулировка «байты состояния на токен» смешивает их и запрещена: для
softmax-внимания это второе число, а для всего остального — бессмысленная
величина `1/N`.

**И счёт где-то всё равно приходит.** Состояние `d_k×d_v` не может хранить
ассоциации `N` токенов — это теория информации, а не задача оптимизации, и
лучшее обучение этого не исправит.

### `Шаг 7. Домашнее задание`

*Нужна ячейка 1.*

**Вопрос.** Сколько ассоциаций «ключ — значение» различает слой линейного
внимания и как это число зависит от $d_k$ и от правила обновления?

Вспомним чтение состояния запросом $q = k_i$:

$$S_t^{\top}k_i=\sum_j (k_j^{\top}k_i)\,v_j=(k_i^{\top}k_i)\,v_i+\underbrace{\sum_{j\ne i}(k_j^{\top}k_i)\,v_j}_{\text{помеха}}$$

Первое слагаемое — нужное значение, сумма — перекрёстные помехи от остальных
записей. Результат определяют два обстоятельства:

- **слой записывает внешнее произведение на каждом токене**, а не только на
  токенах пар. Для последовательности с $M$ парами, $D$ дистракторами и $Q$
  запросами число записей $W = 2M + D + Q$, и прямая теории —
  $M^{\ast} = c\,d_k - (D + Q - 1)/2$;
- **аппроксимация строится со свободным членом.** Подгонка через начало
  координат при $D > 0$ даёт заниженное $c$, смещение уменьшается с ростом
  $d_k$, и на рисунке это выглядит как сублинейный рост.

Вывод $\mathrm{SNR} = d_k/(W - 1)$ и этой прямой — часть 1 задания. Аккуратно
выведенная гипотеза, которую эксперимент опровергает, изложенная вместе с
данными, получает полный балл.

Формулировка, порядок работы и критерии оценивания — в файле `assignment.pdf`.
Каркас ставится из публичного репозитория
[linear-attention-public](https://github.com/voorhs/linear-attention-public).

В отчёте для каждой ветви приводятся два числа состояния: **байты состояния на
слой при длине контекста $N$** и **прирост байтов на токен, на слой**. Ячейка
ниже считает их для конфигурации этого семинара.

In [ ]:
# два числа, которые вы укажете в отчёте, для сегодняшней конфигурации
layer = bench.LinearAttentionLayer(heads=H, d_k=D_K, d_v=D_V, dtype_bytes=4)
by_hand = H * D_K * D_V * 4
assert layer.state_bytes(0) == by_hand == layer.state_bytes(10 ** 6)
print(f"байты состояния на слой при N : H·d_k·d_v·dtype_bytes = "
      f"{H}·{D_K}·{D_V}·4 = {by_hand:,} B = {bench.format_bytes(by_hand)}  (при ЛЮБОМ N)")
print(f"прирост байтов на токен       : {layer.marginal_bytes_per_token()} B")
print("\nдо встречи на следующей неделе.")